# Partie 2 : Présentation InfluxDB

### Organisation des données : 

Les données sont stockées dans un bucket. Chaque point est de la forme suivante : 

`measurement, tag_key=tag_value field_key=field_value timestamp`

Exemple : 

`methode_1,room=chambre temperature=20.1`

Quelques précisions : 
* Bucket : où sont stockées les données
    * Measurement : groupe de données, équivalent d'une table
        * Tags : metadata sur laquelle on peut filtrer - ex : location
        * Fields : valeurs mesurées - ex : temperature
        * Timestamp : date/heure du point, permet d'indexer les valeurs au cours du temps

Utilisons maintenant cela pour écrire des données...

### Méthode 1 : Avec l'UI d'InfluxDB 

Connectez vous sur http://localhost:8086 et remplissez les informations de connexion notées dans les .txt correspondant.

1. Aller dans Load Data -> Buckets
2. Cliquer sur le bucket (home)
3. Cliquer sur "Add Data" -> "Line Protocol" -> "Enter manually"
4. Entrer la ligne suivante :

```bash
methode_1, room=chambre temperature=20.1
```

5. Cliquer sur Write Data

Maintenant, aller dans Data Explorer. 
Sélectionner le bucket *home* et dans le filtre *_measurement* se trouvera *methode_1* !

### Méthode 2 : CLI

Exécuter cette commande pour lancer un terminal bash : (commande `exit` pour sortir)

```bash
docker exec -it influxdb_tuto-influxdb2-1 bash
```

Puis exécuter la commande suivante afin d'ajouter un point directement depuis la CLI :
```bash
influx write --bucket "home" --org "docs"  --token "MyInitialAdminToken0=="  --precision s "methode_2,room=salon temperature=22.5"
```

Retourner dans Data Explorer sur le dashboard. 
Sélectionner le bucket *home* et dans le filtre *_measurement*, il y aura *methode_2* !

### Méthode 3 : Python

Pour ajouter des points depuis Python, on utilise le module *influxdb_client*.

1. Se connecter au client 
2. Définir un Point avec le *measurement*, les *tags*, les *fields* et les *timestamps*
3. Ecrire ensuite le tout dans le bucket

In [ ]:
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
from datetime import datetime

url = "http://influxdb2:8086"
token = open("/run/secrets/influxdb2-admin-token").read().strip()
org = "docs"
bucket = "home"

## Création du client
client = InfluxDBClient(url=url, token=token, org=org)
write_api = client.write_api(write_options=SYNCHRONOUS)

## Définition du point
point = (
    Point("methode_3")
    .tag("room", "cuisine")
    .field("temperature", 23.5)
    .time(datetime.utcnow(), WritePrecision.NS)
)

## Ecriture de la donnée dans le bucket
write_api.write(bucket=bucket, org=org, record=point)

print("Données ajoutées !")

Comme pour les deux méthodes précédentes, retourner dans Data Explorer. \
Sélectionner le bucket *home* et dans le filtre *_measurement*, il y aura *methode_3* !